In [23]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cholesky
import os
import sys

sys.path.append(os.path.abspath(".."))  # Adds the parent directory to sys.path

from privateggm.model import optimize_lbfgs, mirror_descent
from privateggm.utils import *

In [28]:
# generate a toy 3x3 covariance example
def generate_toy_example(Sigma_true, inds, noise_scale):
    """
    Generate a small 3x3 synthetic covariance matrix and parameters for testing.
    """

    # generate symmetric noise matrix
    lower_triangle_noise = np.tril(np.random.normal(0, noise_scale, Sigma_true.shape))
    tau = lower_triangle_noise + lower_triangle_noise.T - np.diag(np.diag(lower_triangle_noise))
    # perturb Sigma_true
    Sigma_tilde = Sigma_true + tau

    # select indices of the observed entries (only diagonal and lower triangle indices)
    rows, cols = inds[:, 0], inds[:, 1]
    Sigma_tilde_vec = Sigma_tilde[rows, cols]

    # Show mask of observed entries
    Sigma_mask = np.zeros_like(Sigma_tilde)
    for i, j in inds:
        Sigma_mask[i, j] = 1
        Sigma_mask[j, i] = 1  # Ensure symmetry if needed
    print(f"Observed entries mask:\n{Sigma_mask}")

    z_init = initialize_z(Sigma_tilde, inds)

    # For mirror descent
    Sigma_init = np.eye(Sigma_tilde.shape[0]) 

    return  Sigma_tilde, Sigma_tilde_vec, z_init, Sigma_init, tau

def test_lbfgs(Sigma_true, inds, noise_scale, max_iters=1000, verbose=False):
    """
    Run the L-BFGS optimization on the toy example.
    """
    Sigma_tilde, Sigma_tilde_vec, z_init, _, _ = generate_toy_example(Sigma_true, inds, noise_scale)
    
    d = Sigma_true.shape[0]  # Dimensionality
    
    # Run lbfgs optimization
    z_opt, g_vals, obj_vals = optimize_lbfgs(Sigma_tilde_vec, z_init, inds, d, max_iters=max_iters, verbose=verbose)
    
    # Convert optimized z into a precision matrix K
    K_z_opt = get_K(z_opt, inds, d)
    
    # Convert back to covariance matrix
    Sigma_opt = np.linalg.inv(K_z_opt)
    
    # Print results
    print(f"\nTrue Covariance Matrix Sigma:\n{Sigma_true}")
    print(f"\nNoisy Covariance Matrix Sigma:\n{Sigma_tilde}")
    print(f"\nOptimized Covariance Matrix Sigma:\n{Sigma_opt}")
    
    # Compute total MSE
    mse_total = np.mean((Sigma_true - Sigma_opt) ** 2)
    
    # Compute MSE only over observed entries
    observed_errors = (Sigma_true[inds[:, 0], inds[:, 1]] - Sigma_opt[inds[:, 0], inds[:, 1]]) ** 2
    mse_observed = np.mean(observed_errors)
    
    # Print error metrics
    print(f"\nMSE (total) between true and optimized Sigma: {mse_total:.6f}")
    print(f"\nMSE (observed entries only) between true and optimized Sigma: {mse_observed:.6f}")

def test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=20000, eta_init=0.2, tol=1e-8, verbose=True):

    # Generate toy dataset
    Sigma_tilde, _, _, Sigma_init, tau = generate_toy_example(Sigma_true, inds, noise_scale)
    
    # Run Mirror Descent
    Sigma_opt, K_opt = mirror_descent(Sigma_init, Sigma_tilde, inds, tau, 
                                      max_iters=max_iters, eta_init=eta_init, tol=tol, verbose=verbose)
    
    # Print results
    print(f"\nTrue Covariance Matrix Sigma:\n{Sigma_true}")
    print(f"\nNoisy Covariance Matrix Sigma:\n{Sigma_tilde}")
    print(f"\nOptimized Covariance Matrix Sigma (Mirror Descent):\n{Sigma_opt}")
    
    # Compute total MSE
    mse_total = np.mean((Sigma_true - Sigma_opt) ** 2)
    
    # Compute MSE only over observed entries
    observed_errors = (Sigma_true[inds[:, 0], inds[:, 1]] - Sigma_opt[inds[:, 0], inds[:, 1]]) ** 2
    mse_observed = np.mean(observed_errors)
    
    # Print error metrics
    print(f"\nMSE (total) between true and optimized Sigma: {mse_total:.6f}")
    print(f"\nMSE (observed entries only) between true and optimized Sigma: {mse_observed:.6f}")


# 3 x 3 case

In [25]:
np.random.seed(42)

# Define a well-conditioned covariance matrix
d = 3
Sigma_true = np.array([[2.0, 0.5, 0.3],
                       [0.5, 1.5, 0.4],
                       [0.3, 0.4, 1.0]])
noise_scale = 0.1

inds = np.array([[0,0], [1,1], [2,2], [1,0], [2,0]])

test_lbfgs(Sigma_true, inds, noise_scale, max_iters=2000, verbose=True)
test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=2000, eta_init=0.01, tol=1e-6, verbose=True)

Observed entries mask:
[[1. 1. 1.]
 [1. 1. 0.]
 [1. 0. 1.]]
Iteration 0: Loss = 4.059326e+00, Grad Sum = 2.220449e+00, Step Size = 1.000000e+00
Iteration 1: Loss = 3.862610e+00, Grad Sum = 1.445340e+00, Step Size = 1.000000e+00
Iteration 2: Loss = 3.804332e+00, Grad Sum = 7.928121e-01, Step Size = 1.000000e+00
Iteration 3: Loss = 3.796853e+00, Grad Sum = 2.238699e-01, Step Size = 1.000000e+00
Iteration 4: Loss = 3.795656e+00, Grad Sum = 1.617817e-01, Step Size = 1.000000e+00
Iteration 5: Loss = 3.794361e+00, Grad Sum = 8.152925e-02, Step Size = 1.000000e+00
Iteration 6: Loss = 3.794288e+00, Grad Sum = 4.017826e-02, Step Size = 1.000000e+00
Iteration 7: Loss = 3.794263e+00, Grad Sum = 7.769105e-03, Step Size = 1.000000e+00
Iteration 8: Loss = 3.794262e+00, Grad Sum = 4.146353e-03, Step Size = 1.000000e+00
Iteration 9: Loss = 3.794261e+00, Grad Sum = 2.837188e-04, Step Size = 1.000000e+00
Optimization converged at iteration 9: Loss Change = 6.2042490e-07

True Covariance Matrix Sigma:
[[

# Pathological case (d=10)

In [30]:
np.random.seed(42)

# Load problematic Sigma_obs and set it as Sigma_true
Sigma_true = np.load("../pathologic_d10_sigma_tilde_dp.npy")

# Fetch the observed indices
inds = get_observed_Sigma_entries(Sigma_true)

# Noise scale needs to be 0 (we want to optimize the pathologic case directly)
noise_scale = 0.

test_lbfgs(Sigma_true, inds, noise_scale, max_iters=2000, verbose=True)
test_mirror_descent(Sigma_true, inds, noise_scale, max_iters=2000, eta_init=0.01, tol=1e-6, verbose=True)

Observed entries mask:
[[1. 0. 0. 1. 0. 1. 0. 1. 1. 0.]
 [0. 1. 1. 0. 1. 1. 1. 1. 1. 0.]
 [0. 1. 1. 1. 1. 0. 1. 1. 1. 0.]
 [1. 0. 1. 1. 0. 0. 0. 1. 0. 0.]
 [0. 1. 1. 0. 1. 0. 1. 0. 1. 1.]
 [1. 1. 0. 0. 0. 1. 1. 0. 0. 1.]
 [0. 1. 1. 0. 1. 1. 1. 0. 0. 0.]
 [1. 1. 1. 1. 0. 0. 0. 1. 0. 1.]
 [1. 1. 1. 0. 1. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1. 1. 0. 1. 0. 1.]]
Iteration 0: Loss = -1.600155e+01, Grad Sum = 1.579044e+00, Step Size = 1.000000e+00
Iteration 1: Loss = -1.847682e+01, Grad Sum = 1.345471e+00, Step Size = 1.000000e+00
Iteration 2: Loss = -2.322021e+01, Grad Sum = 5.546664e+00, Step Size = 1.000000e+00
Iteration 3: Loss = -2.471207e+01, Grad Sum = 1.097356e+00, Step Size = 1.000000e+00
Iteration 4: Loss = -2.679790e+01, Grad Sum = 1.244644e+00, Step Size = 1.000000e+00
Iteration 5: Loss = -2.993589e+01, Grad Sum = 2.533816e+00, Step Size = 1.000000e+00
Iteration 6: Loss = -3.523056e+01, Grad Sum = 2.772496e+00, Step Size = 1.000000e+00
Iteration 7: Loss = -4.404611e+01, Grad Sum = 1.100

/Users/ceciliaferrando/Documents/UMASS/RESEARCH/GaussianPGM/private-ggm/privateggm/model.py:174: RuntimeWarning: divide by zero encountered in scalar divide
  grad[i,j] = (2/tau[i,j]**2) * (Sigma[i,j] - Sigma_tilde[i,j])
/Users/ceciliaferrando/Documents/UMASS/RESEARCH/GaussianPGM/private-ggm/privateggm/model.py:165: RuntimeWarning: divide by zero encountered in scalar divide
  loss += (1 / tau[i, j] ** 2) * (Sigma_tilde[i, j] - Sigma[i, j]) ** 2


ValueError: array must not contain infs or NaNs

# ADMM attempt (DeepSeek)

In [18]:
def estimate_precision_matrix_admm(
        observed_covariance: np.ndarray, 
        observation_mask: np.ndarray, 
        entropy_weight: float, 
        penalty_param: float = 1.0, 
        max_iterations: int = 1000,
        convergence_tol: float = 1e-6,
        psd_jitter: float = 1e-8) -> np.ndarray:
    """
    ADMM solver for maximum entropy precision matrix estimation with explicit variable splitting.
    
    Splits the problem into two subproblems:
    1. Covariance estimation (data fidelity)
    2. Precision matrix estimation (entropy maximization + PSD constraints)
    
    Parameters:
        observed_covariance (np.ndarray): Noisy covariance matrix with NaNs for missing entries
        observation_mask (np.ndarray): Boolean mask indicating observed entries
        entropy_weight (float): Weight for entropy regularization (\u03bb in -\u03bb logdet term)
        penalty_param (float): Initial ADMM penalty parameter (\u03c1)
        max_iterations (int): Maximum number of ADMM iterations
        convergence_tol (float): Tolerance for primal/dual residual convergence
        psd_jitter (float): Small value to ensure positive definiteness
        
    Returns:
        np.ndarray: Estimated precision matrix \u0398 = \u03a3^{-1}
    """
    
    # Problem dimensions
    n = observed_covariance.shape[0]
    observed_entries = observation_mask.astype(bool)
    
    # ADMM variables with explicit roles
    covariance_estimate = np.eye(n, dtype=np.float64)  # \u03a3: Data fidelity estimate
    precision_estimate = np.eye(n, dtype=np.float64)   # Z: PSD/Entropy-constrained estimate
    dual_variable = np.zeros((n, n), dtype=np.float64)  # U: Lagrange multiplier for \u03a3-Z=0
    
    # Track residuals for adaptive penalty adjustment
    primal_residual_history = []
    dual_residual_history = []
    
    # ADMM iteration
    for iteration in range(max_iterations):
        prev_precision = precision_estimate.copy()
        
        # --- Covariance Update (Data Fidelity) ---
        numerator = np.where(observed_entries, observed_covariance, 0.0) 
        numerator += penalty_param * (precision_estimate - dual_variable)
        denominator = np.where(observed_entries, 1.0, 0.0) + penalty_param
        covariance_estimate = numerator / denominator
        
        # Project to PSD cone
        eigenvalues, eigenvectors = np.linalg.eigh(covariance_estimate)
        eigenvalues = np.maximum(eigenvalues, psd_jitter)
        covariance_estimate = eigenvectors @ np.diag(eigenvalues) @ eigenvectors.T
        covariance_estimate = (covariance_estimate + covariance_estimate.T) / 2  

        # --- Precision Matrix Update (Entropy + PSD) ---
        constraint_matrix = covariance_estimate + dual_variable
        constraint_matrix = (constraint_matrix + constraint_matrix.T) / 2  
        
        eigenvalues, eigenvectors = np.linalg.eigh(constraint_matrix)
        adjusted_eigenvalues = (eigenvalues + np.sqrt(eigenvalues**2 + 4*entropy_weight/penalty_param)) / 2
        precision_estimate = eigenvectors @ np.diag(adjusted_eigenvalues) @ eigenvectors.T

        # --- Dual Variable Update (Constraint Enforcement) ---
        dual_variable += covariance_estimate - precision_estimate
        dual_variable = (dual_variable + dual_variable.T) / 2  

        # --- Convergence Check ---
        primal_residual = np.linalg.norm(covariance_estimate - precision_estimate, 'fro')
        dual_residual = penalty_param * np.linalg.norm(precision_estimate - prev_precision, 'fro')
        
        if primal_residual > 10 * dual_residual:
            penalty_param *= 2
            dual_variable /= 2  
        elif dual_residual > 10 * primal_residual:
            penalty_param /= 2
            dual_variable *= 2
            
        if primal_residual < convergence_tol and dual_residual < convergence_tol:
            break

    return np.linalg.inv(precision_estimate + psd_jitter*np.eye(n))

# Example Usage ----------------------------------------------------------------

n = 50
true_precision = np.eye(n) + 0.1*(np.random.rand(n, n) < 0.05).astype(float)
true_precision = (true_precision + true_precision.T)/2  
true_covariance = np.linalg.inv(true_precision)

observation_mask = np.random.rand(n, n) < 0.3
observation_mask = np.triu(observation_mask) | np.triu(observation_mask, 1).T  

observed_covariance = true_covariance.copy()
observed_covariance[observation_mask] += np.random.normal(0, 0.1, np.sum(observation_mask))
observed_covariance = np.where(observation_mask, observed_covariance, np.nan)

theta_est = estimate_precision_matrix_admm(
    observed_covariance=observed_covariance,
    observation_mask=observation_mask,
    entropy_weight=0.1,
    penalty_param=1.0,
    max_iterations=1000
)

assert np.all(np.linalg.eigvals(theta_est) > 0), "Precision matrix must be positive definite!"
print("ADMM converged to valid precision matrix")

ADMM converged to valid precision matrix


In [21]:
def test_admm():
    """
    Test the ADMM function on a small 3x3 covariance example.
    """
    # Define a well-conditioned covariance matrix
    d = 3
    Sigma_true = np.array([[2.0, 0.5, 0.3],
                           [0.5, 1.5, 0.4],
                           [0.3, 0.4, 1.0]])
    noise_scale = 0.1

    # Define observed indices mask
    inds = np.array([[0,0], [1,1], [2,2], [1,0], [2,0]])
    observation_mask = np.zeros((d, d), dtype=bool)
    for i, j in inds:
        observation_mask[i, j] = True
        observation_mask[j, i] = True  # Ensure symmetry

    # Create a copy of Sigma_true but with noise only on observed entries
    observed_covariance = Sigma_true.copy()
    
    # Add noise only to observed entries (without modifying NaN positions)
    noise_matrix = np.random.normal(0, noise_scale, (d, d))
    noise_matrix = (noise_matrix + noise_matrix.T) / 2  # Ensure symmetry
    observed_covariance[observation_mask] += noise_matrix[observation_mask]

    # Run ADMM solver
    entropy_weight = 0.1  # λ for entropy regularization
    penalty_param = 1.0   # ρ for ADMM penalty parameter

    theta_est = estimate_precision_matrix_admm(
        observed_covariance=observed_covariance,
        observation_mask=observation_mask,
        entropy_weight=entropy_weight,
        penalty_param=penalty_param,
        max_iterations=1000
    )

    # Compute errors
    mse_total = np.mean((Sigma_true - np.linalg.inv(theta_est)) ** 2)
    observed_errors = (Sigma_true[observation_mask] - np.linalg.inv(theta_est)[observation_mask]) ** 2
    mse_observed = np.mean(observed_errors)

    # Print results
    print(f"\nTrue Covariance Matrix Sigma:\n{Sigma_true}")
    print(f"\nNoisy Covariance Matrix Sigma:\n{observed_covariance}")
    print(f"\nEstimated Covariance Matrix (ADMM):\n{np.linalg.inv(theta_est)}")
    print(f"\nMSE (total) between true and estimated Sigma: {mse_total:.6f}")
    print(f"\nMSE (observed entries only) between true and estimated Sigma: {mse_observed:.6f}")

# Run test
test_admm()


True Covariance Matrix Sigma:
[[2.  0.5 0.3]
 [0.5 1.5 0.4]
 [0.3 0.4 1. ]]

Noisy Covariance Matrix Sigma:
[[1.90251934 0.41626361 0.36109162]
 [0.41626361 1.57826298 0.4       ]
 [0.36109162 0.4        0.68232962]]

Estimated Covariance Matrix (ADMM):
[[1.96018465 0.40307772 0.33827059]
 [0.40307772 1.6423861  0.06955149]
 [0.33827059 0.06955149 0.81457065]]

MSE (total) between true and estimated Sigma: 0.032928

MSE (observed entries only) between true and estimated Sigma: 0.011137
